# 🎯 Notebook 03 — Talent Score ML

**Phase 5 du projet KCorp Scouting Tool**

Ce notebook documente le pipeline de modélisation supervisée pour prédire `promoted_to_lec`.
Il est conçu comme un journal de bord : chaque décision technique est expliquée et justifiée.

## Sommaire
1. [Contexte & Problématique](#1-contexte)
2. [Chargement et split Out-of-Time](#2-split)
3. [Entraînement des modèles (LR, RF, XGBoost)](#3-training)
4. [Tuning du Random Forest (RandomizedSearchCV)](#4-tuning)
5. [Évaluation comparée — PR-AUC](#5-evaluation)
6. [Feature Importances](#6-features)
7. [Distribution des Talent Scores](#7-distribution)
8. [Leaderboard des pépites](#8-leaderboard)
9. [Conclusions & limites](#9-conclusions)

## 0. Imports

In [ ]:
import sys
import os
sys.path.insert(0, os.path.abspath('..'))

from IPython.display import display
import json
import warnings
warnings.filterwarnings('ignore')

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import joblib
from pathlib import Path
from sklearn.metrics import average_precision_score, roc_auc_score, precision_recall_curve

# Chemins
ROOT = Path('..').resolve() if Path.cwd().name == 'notebooks' else Path('.').resolve()
METRICS_DIR = ROOT / 'reports' / 'metrics'
FIGURES_DIR = ROOT / 'reports' / 'figures'
MODELS_DIR  = ROOT / 'models'

plt.style.use('dark_background')
plt.rcParams.update({'font.family': 'DejaVu Sans', 'figure.facecolor': '#0f172a',
                     'axes.facecolor': '#1e293b', 'text.color': '#f1f5f9'})

print('Imports OK')

---
## 1. Contexte & Problématique <a id='1-contexte'></a>

### Pourquoi ce problème est difficile

Identifier les futurs joueurs LEC parmi les ERL est un problème de classification **très déséquilibré** :
- ~2 000 joueurs/splits ERL dans notre dataset
- ~200 promus LEC → **≈ 10% de positifs**
- Un modèle qui prédit "jamais promu" pour tout le monde aurait **90% d'accuracy** → inutile

### Nos choix méthodologiques

| Choix | Justification |
|---|---|
| **Out-of-Time split** | Évite le data leakage temporel |
| **PR-AUC** comme métrique | Adaptée aux classes très déséquilibrées |
| **Z-scores intra-ligue** | Compare les joueurs à leurs pairs, pas au niveau absolu |
| **class_weight='balanced'** | Compense le déséquilibre dans la loss |

---
## 2. Chargement et Split Out-of-Time <a id='2-split'></a>

In [ ]:
from src.models.talent_scorer import (
    load_features, make_out_of_time_split, FEATURE_COLS
)

df = load_features()
X_train, y_train, X_test, y_test, df_train, df_test = make_out_of_time_split(df)
available_features = [c for c in FEATURE_COLS if c in X_train.columns]

print(f'Train 2024 : {len(X_train):,} lignes | {y_train.sum()} promus ({y_train.mean():.1%})')
print(f'Test  2025 : {len(X_test):,} lignes | {y_test.sum()} promus ({y_test.mean():.1%})')
print(f'\nFeatures utilisées ({len(available_features)}) :')
for f in available_features:
    print(f'  {f}')

In [ ]:
# Visualisation du déséquilibre
fig, axes = plt.subplots(1, 2, figsize=(10, 4))
fig.suptitle('Déséquilibre des classes — Train vs Test', fontweight='bold')

for ax, (split_name, y) in zip(axes, [('Train 2024', y_train), ('Test 2025', y_test)]):
    ax.set_facecolor('#1e293b')
    counts = y.value_counts()
    bars = ax.bar(['Non promu', 'Promu LEC'],
                  [counts.get(0, 0), counts.get(1, 0)],
                  color=['#64748b', '#facc15'])
    for bar in bars:
        ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 5,
                f'{int(bar.get_height())}', ha='center', fontsize=11, fontweight='bold')
    ax.set_title(split_name, fontweight='bold')
    ax.set_ylabel('Nombre de joueurs/splits')

plt.tight_layout()
plt.show()

---
## 3. Entraînement des modèles <a id='3-training'></a>

In [ ]:
from src.models.talent_scorer import (
    build_logistic_regression, build_random_forest, build_xgboost, evaluate_model
)

# Calcul du scale_pos_weight XGBoost
neg_count = (y_train == 0).sum()
pos_count = (y_train == 1).sum()
scale_pos_weight = neg_count / pos_count
print(f'scale_pos_weight XGBoost : {scale_pos_weight:.1f}')

models_base = {
    'Logistic Regression': build_logistic_regression(),
    'Random Forest':       build_random_forest(),
    'XGBoost':             build_xgboost(),
}
models_base['XGBoost'].set_params(scale_pos_weight=scale_pos_weight)

results = {}
for name, model in models_base.items():
    print(f'  Entraînement : {name}...')
    model.fit(X_train, y_train)
    y_proba = model.predict_proba(X_test)[:, 1]
    results[name] = {
        'model': model,
        'y_proba': y_proba,
        'pr_auc': average_precision_score(y_test, y_proba),
        'roc_auc': roc_auc_score(y_test, y_proba),
    }
    print(f'    PR-AUC={results[name]["pr_auc"]:.4f}  ROC-AUC={results[name]["roc_auc"]:.4f}')

print('\nModèles entraînés.')

---
## 4. Tuning du Random Forest <a id='4-tuning'></a>

**Stratégie** : RandomizedSearchCV avec 60 itérations et 5-fold stratifié, optimisant la PR-AUC.

> **Résultat observé** : Le RF tuné (PR-AUC = 0.182) est *moins bon* que le RF par défaut (0.190).
> Cela révèle que la relation entre les features et la promotion est essentiellement **linéaire** avec
> seulement 739 exemples — la Régression Logistique l'exploite mieux sans sur-paramétrage.

In [ ]:
# Chargement des résultats de tuning déjà calculés
with open(METRICS_DIR / 'talent_score_results.json', encoding='utf-8') as f:
    saved = json.load(f)

print(f'Meilleurs hyperparamètres RF (RandomizedSearchCV) :')
for param, val in saved.get('rf_tuned_params', {}).items():
    print(f'  {param:<25} = {val}')
print(f'\nCV PR-AUC : {saved.get("rf_tuned_cv_pr_auc", "N/A")}')

---
## 5. Évaluation comparée — Courbes PR <a id='5-evaluation'></a>

In [ ]:
# Chargement des résultats sauvegardés (inclut le RF tuné)
comparison = pd.DataFrame(saved['comparison']).sort_values('pr_auc', ascending=False)
display(comparison[['model_name', 'pr_auc', 'roc_auc']].round(4))

In [ ]:
# Affichage des figures déjà générées
from IPython.display import Image, display as ipy_display

print('Courbes Précision-Rappel :')
ipy_display(Image(str(FIGURES_DIR / '01_pr_curves.png'), width=800))

In [ ]:
print('Comparaison des métriques :')
ipy_display(Image(str(FIGURES_DIR / '02_metrics_comparison.png'), width=900))

---
## 6. Feature Importances <a id='6-features'></a>

Les importances sont calculées différemment selon le modèle :
- **Logistic Regression** : coefficients absolus (impact directionnel sur log-odds)
- **Random Forest / XGBoost** : gain moyen pondéré (réduction d'impureté)

In [ ]:
from IPython.display import Image, display as ipy_display
ipy_display(Image(str(FIGURES_DIR / '03_feature_importances.png'), width=900))

In [ ]:
# Feature importances LR (coefficients)
lr_fi = pd.read_csv(METRICS_DIR / 'feature_importance_logistic_regression_baseline_.csv')
print('Feature importances — Logistic Regression (|coef| normalisé) :')
display(lr_fi.head(10).round(4))

### Interprétation

- **`games_played` (31.8% en LR)** : Un joueur titulaire joue plus → signal de confiance du coach
- **`dpm_zscore` (9.7%)** : Les dégâts par minute relatifs à la ligue — performance offensive
- **`champion_pool_size_zscore` (11.6%)** : La polyvalence — les joueurs LEC doivent s'adapter
- **`golddiffat15_zscore` (7.9%)** : Domination en early game — crucial en compétition haut niveau

---
## 7. Distribution des Talent Scores <a id='7-distribution'></a>

In [ ]:
from IPython.display import Image, display as ipy_display
ipy_display(Image(str(FIGURES_DIR / '04_score_distribution.png'), width=900))

In [ ]:
scores_df = pd.read_csv(METRICS_DIR / 'talent_scores_players.csv')
promoted = scores_df[scores_df['promoted_to_lec'] == True]
not_promoted = scores_df[scores_df['promoted_to_lec'] == False]

print(f'Médiane scores — Promus    : {promoted["talent_score"].median():.1f}')
print(f'Médiane scores — Non promus: {not_promoted["talent_score"].median():.1f}')
print(f'Séparation : {promoted["talent_score"].median() - not_promoted["talent_score"].median():.1f} points')

---
## 8. Leaderboard des pépites <a id='8-leaderboard'></a>

In [ ]:
from IPython.display import Image, display as ipy_display
ipy_display(Image(str(FIGURES_DIR / '05_leaderboard_top30.png'), width=1000))

In [ ]:
# Top 15 pépites — meilleur score par joueur
top = (
    scores_df
    .sort_values('talent_score', ascending=False)
    .drop_duplicates(subset=['playername'])
    .head(15)
    [['playername', 'league', 'position', 'talent_score', 'promoted_to_lec', 'win_rate', 'games_played']]
    .reset_index(drop=True)
)
top.index += 1
display(top)

In [ ]:
# Leaderboard par ligue
from IPython.display import Image, display as ipy_display
print('Top 5 par ligue :')
ipy_display(Image(str(FIGURES_DIR / '06_leaderboard_by_league.png'), width=1000))

---
## 9. Conclusions & Limites <a id='9-conclusions'></a>

### Ce qu'on a appris

1. **La Régression Logistique est le meilleur modèle** (PR-AUC = 0.256) — la relation entre les Z-scores et la promotion est essentiellement linéaire. Les modèles plus complexes (RF, XGBoost) sur-apprennent sur les ~740 exemples du train set.

2. **Le signal existe** : médiane promus = 58/100 vs non-promus = 21/100. Le modèle classe correctement 24% des vrais promus dans le Top-104 (contre 10.6% pour un classement aléatoire).

3. **`games_played` est le signal le plus fort** : le nombre de matchs joués capture la régularité et la confiance du coach — un signal esport très pertinent.

### Limites identifiées

- **Dataset petit** : 739 exemples d'entraînement dont seulement 62 positifs — la puissance statistique est limitée.
- **Promotions non-méritocratiques** : certains joueurs sont promus pour des raisons contractuelles/économiques, pas sur leurs performances ERL.
- **Pas de features de progression** : l'évolution d'un joueur entre deux splits serait un signal fort non capturé ici.

### Prochaines étapes
→ **Notebook 04** : Clustering par style de jeu pour identifier les profils "LEC-ready"